In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

2026-06-17 04:29:12.715680: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781670552.915074      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781670552.970498      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781670553.444076      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781670553.444119      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781670553.444122      58 computation_placer.cc:177] computation placer alr

In [2]:
DATASET_PATH = "/kaggle/input/datasets/sauravmane9440/skin-cancer-dataset/final_dataset"

IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 20

In [3]:
image_paths = []
mask_paths = []

classes = [
    "abrasion",
    "akiec",
    "bcc",
    "bkl",
    "bruise",
    "burn",
    "cut",
    "df",
    "mel",
    "normal",
    "nv",
    "vasc"
]

for cls in classes:

    img_dir = os.path.join(
        DATASET_PATH,
        cls
    )

    mask_dir = os.path.join(
        DATASET_PATH,
        "mask",
        cls
    )

    for file in os.listdir(img_dir):

        img_path = os.path.join(
            img_dir,
            file
        )

        mask_path = os.path.join(
            mask_dir,
            file
        )

        if os.path.exists(mask_path):

            image_paths.append(
                img_path
            )

            mask_paths.append(
                mask_path
            )

print("Total Images :", len(image_paths))

Total Images : 20195


In [4]:
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.30,
    random_state=42
)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs,
    temp_masks,
    test_size=0.50,
    random_state=42
)

print("Train :", len(train_imgs))
print("Val   :", len(val_imgs))
print("Test  :", len(test_imgs))

Train : 14136
Val   : 3029
Test  : 3030


In [5]:
from tensorflow.keras.utils import Sequence

class UNetGenerator(Sequence):

    def __init__(
        self,
        image_paths,
        mask_paths,
        batch_size=8,
        shuffle=True
    ):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.shuffle = shuffle

        self.indices = np.arange(
            len(self.image_paths)
        )

        self.on_epoch_end()

    def __len__(self):

        return int(
            np.ceil(
                len(self.image_paths)
                / self.batch_size
            )
        )

    def __getitem__(self, index):

        batch_indices = self.indices[
            index*self.batch_size:
            (index+1)*self.batch_size
        ]

        X = np.zeros(
            (
                len(batch_indices),
                IMG_SIZE,
                IMG_SIZE,
                3
            ),
            dtype=np.float32
        )

        Y = np.zeros(
            (
                len(batch_indices),
                IMG_SIZE,
                IMG_SIZE,
                1
            ),
            dtype=np.float32
        )

        for i, idx in enumerate(batch_indices):

            img = cv2.imread(
                self.image_paths[idx]
            )

            img = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2RGB
            )

            img = cv2.resize(
                img,
                (IMG_SIZE, IMG_SIZE)
            )

            mask = cv2.imread(
                self.mask_paths[idx],
                0
            )

            mask = cv2.resize(
                mask,
                (IMG_SIZE, IMG_SIZE)
            )

            X[i] = img.astype(
                np.float32
            ) / 255.0

            Y[i,:,:,0] = (
                mask > 127
            ).astype(np.float32)

        return X, Y

    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(
                self.indices
            )

In [6]:
train_gen = UNetGenerator(
    train_imgs,
    train_masks,
    batch_size=BATCH_SIZE
)

val_gen = UNetGenerator(
    val_imgs,
    val_masks,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_gen = UNetGenerator(
    test_imgs,
    test_masks,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [7]:
def dice_coef(
    y_true,
    y_pred
):

    smooth = 1e-6

    y_true_f = tf.keras.backend.flatten(
        y_true
    )

    y_pred_f = tf.keras.backend.flatten(
        y_pred
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    return (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        + smooth
    )


def iou_score(
    y_true,
    y_pred
):

    smooth = 1e-6

    intersection = tf.reduce_sum(
        y_true * y_pred
    )

    union = (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

In [8]:
def build_unet():

    inputs = Input(
        (IMG_SIZE, IMG_SIZE, 3)
    )

    # Encoder

    c1 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(inputs)

    c1 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(c1)

    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(p1)

    c2 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(c2)

    p2 = MaxPooling2D()(c2)

    c3 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(p2)

    c3 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(c3)

    p3 = MaxPooling2D()(c3)

    c4 = Conv2D(
        512,
        3,
        activation="relu",
        padding="same"
    )(p3)

    c4 = Conv2D(
        512,
        3,
        activation="relu",
        padding="same"
    )(c4)

    # Decoder

    u1 = UpSampling2D()(c4)

    u1 = Concatenate()(
        [u1, c3]
    )

    c5 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(u1)

    c5 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(c5)

    u2 = UpSampling2D()(c5)

    u2 = Concatenate()(
        [u2, c2]
    )

    c6 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(u2)

    c6 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(c6)

    u3 = UpSampling2D()(c6)

    u3 = Concatenate()(
        [u3, c1]
    )

    c7 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(u3)

    c7 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(c7)

    outputs = Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c7)

    return Model(
        inputs,
        outputs
    )

In [9]:
def build_unet():

    inputs = Input(
        (IMG_SIZE, IMG_SIZE, 3)
    )

    # Encoder

    c1 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(inputs)

    c1 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(c1)

    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(p1)

    c2 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(c2)

    p2 = MaxPooling2D()(c2)

    c3 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(p2)

    c3 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(c3)

    p3 = MaxPooling2D()(c3)

    c4 = Conv2D(
        512,
        3,
        activation="relu",
        padding="same"
    )(p3)

    c4 = Conv2D(
        512,
        3,
        activation="relu",
        padding="same"
    )(c4)

    # Decoder

    u1 = UpSampling2D()(c4)

    u1 = Concatenate()(
        [u1, c3]
    )

    c5 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(u1)

    c5 = Conv2D(
        256,
        3,
        activation="relu",
        padding="same"
    )(c5)

    u2 = UpSampling2D()(c5)

    u2 = Concatenate()(
        [u2, c2]
    )

    c6 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(u2)

    c6 = Conv2D(
        128,
        3,
        activation="relu",
        padding="same"
    )(c6)

    u3 = UpSampling2D()(c6)

    u3 = Concatenate()(
        [u3, c1]
    )

    c7 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(u3)

    c7 = Conv2D(
        64,
        3,
        activation="relu",
        padding="same"
    )(c7)

    outputs = Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c7)

    return Model(
        inputs,
        outputs
    )

In [10]:
def dice_loss(
    y_true,
    y_pred
):
    return 1 - dice_coef(
        y_true,
        y_pred
    )

unet = build_unet()

unet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=dice_loss,
    metrics=[
        "accuracy",
        dice_coef,
        iou_score
    ]
)

I0000 00:00:1781670595.429763      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [11]:
callbacks = [

    ModelCheckpoint(
        "best_unet.keras",
        save_best_only=True,
        monitor="val_dice_coef",
        mode="max"
    ),

    EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        patience=2
    )
]

In [12]:
history = unet.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20


I0000 00:00:1781670602.520724     123 service.cc:152] XLA service 0x7a04c0019000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781670602.520762     123 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1781670603.525674     123 cuda_dnn.cc:529] Loaded cuDNN version 91002


   2/1767 ━━━━━━━━━━━━━━━━━━━━ 1:35 54ms/step - accuracy: 0.0444 - dice_coef: 0.0813 - iou_score: 0.0424 - loss: 0.9187  

I0000 00:00:1781670612.460244     123 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1767/1767 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.9201 - dice_coef: 0.7481 - iou_score: 0.6400 - loss: 0.2519

2026-06-17 04:35:31.074542: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-17 04:35:31.319860: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1767/1767 ━━━━━━━━━━━━━━━━━━━━ 335s 181ms/step - accuracy: 0.9679 - dice_coef: 0.8326 - iou_score: 0.7333 - loss: 0.1674 - val_accuracy: 0.9764 - val_dice_coef: 0.7913 - val_iou_score: 0.6800 - val_loss: 0.2079 - learning_rate: 1.0000e-04
Epoch 2/20
1767/1767 ━━━━━━━━━━━━━━━━━━━━ 187s 106ms/step - accuracy: 0.9843 - dice_coef: 0.8876 - iou_score: 0.8061 - loss: 0.1124 - val_accuracy: 0.9859 - val_dice_coef: 0.8840 - val_iou_score: 0.8055 - val_loss: 0.1151 - learning_rate: 1.0000e-04
Epoch 3/20
1767/1767 ━━━━━━━━━━━━━━━━━━━━ 185s 105ms/step - accuracy: 0.9869 - dice_coef: 0.9050 - iou_score: 0.8331 - loss: 0.0950 - val_accuracy: 0.9869 - val_dice_coef: 0.8992 - val_iou_score: 0.8272 - val_loss: 0.0999 - learning_rate: 1.0000e-04
Epoch 4/20
1767/1767 ━━━━━━━━━━━━━━━━━━━━ 186s 105ms/step - accuracy: 0.9873 - dice_coef: 0.9092 - iou_score: 0.8400 - loss: 0.0908 - val_accuracy: 0.9890 - val_dice_coef: 0.9190 - val_iou_score: 0.8569 - val_loss: 0.0801 - learning_rate: 1.0000e-04
Epoch 5/20


In [13]:
results = unet.evaluate(
    test_gen,
    verbose=1
)

print("Test Loss :", results[0])
print("Test Accuracy :", results[1])
print("Dice Score :", results[2])
print("IoU Score :", results[3])

378/379 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - accuracy: 0.9926 - dice_coef: 0.9445 - iou_score: 0.8965 - loss: 0.0555

2026-06-17 05:36:29.964952: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-17 05:36:30.210539: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


379/379 ━━━━━━━━━━━━━━━━━━━━ 70s 185ms/step - accuracy: 0.9926 - dice_coef: 0.9440 - iou_score: 0.8960 - loss: 0.0560
Test Loss : 0.05599251016974449
Test Accuracy : 0.9926265478134155
Dice Score : 0.9439997673034668
IoU Score : 0.8959853649139404


In [14]:
unet.save(
    "/kaggle/working/unet_model.h5"
)

In [15]:
import tensorflow as tf

def dice_coef(y_true, y_pred):
    smooth = 1e-6
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    return (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        + smooth
    )

def iou_score(y_true, y_pred):
    smooth = 1e-6

    intersection = tf.reduce_sum(y_true * y_pred)

    union = (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

unet = tf.keras.models.load_model(
    "/kaggle/working/unet_model.h5",
    custom_objects={
        "dice_coef": dice_coef,
        "iou_score": iou_score,
        "dice_loss": dice_loss
    }
)

print("Model Loaded Successfully")

Model Loaded Successfully


In [16]:
import os
import cv2
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/sauravmane9440/skin-cancer-dataset/final_dataset"

SAVE_MASK_PATH = "/kaggle/working/unet_masks"

IMG_SIZE = 128

classes = [
    "abrasion","akiec","bcc","bkl",
    "bruise","burn","cut","df",
    "mel","normal","nv","vasc"
]

for cls in classes:

    img_dir = os.path.join(DATASET_PATH, cls)

    save_dir = os.path.join(
        SAVE_MASK_PATH,
        cls
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    files = os.listdir(img_dir)

    print(f"Processing {cls}")

    for file in files:

        img_path = os.path.join(
            img_dir,
            file
        )

        img = cv2.imread(img_path)

        if img is None:
            continue

        img_rgb = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        img_resized = cv2.resize(
            img_rgb,
            (IMG_SIZE, IMG_SIZE)
        )

        x = np.expand_dims(
            img_resized / 255.0,
            axis=0
        )

        pred_mask = unet.predict(
            x,
            verbose=0
        )[0]

        pred_mask = (
            pred_mask[:,:,0] > 0.5
        ).astype(np.uint8)

        cv2.imwrite(
            os.path.join(save_dir,file),
            pred_mask * 255
        )

print("Mask Generation Completed")

Processing abrasion
Processing akiec
Processing bcc
Processing bkl
Processing bruise
Processing burn
Processing cut
Processing df
Processing mel
Processing normal
Processing nv
Processing vasc
Mask Generation Completed


In [17]:
import shutil

shutil.make_archive(
    "/kaggle/working/unet_masks",
    "zip",
    "/kaggle/working/unet_masks"
)

print("ZIP Created")

ZIP Created
